# Lab 0: Qiskit Global Summer School 2026 へようこそ

この最初の行が読めているなら、On24 のイベントに無事参加できているということです。素晴らしい！これは Lab 0、本編に向けて準備するためのウォームアップ用ノートブックです。このラボでは次のことを行います。

- Python パッケージのアップグレード
- Sampler と Estimator を使った簡単なコーディング演習
- 特別なマルチ量子ビット量子回路の構築と、その状態ベクトルの探索
- *（任意）* Qiskit C API を使って C で量子回路を構築

わくわくしますか？この先にはまだたくさんあります！


このラボは macOS、Linux、および [qBraid](https://qbraid.com/) や [Google Colab](https://colab.research.google.com/) のようなクラウドプラットフォームで動作します。Windows を使っている場合は、最良の体験を得るためにクラウドプラットフォームの利用をおすすめします。特に今後のラボでは、一部のパッケージが Linux ベースの環境で最もよく動作します。

以下は簡単なまとめです。

| あなたのOS | このラボの実行方法 |
|---|---|
| macOS / Linux | ローカルで実行 — ノートブックがセットアップを行います。 |
| Windows | 第1〜2章は Python と venv を使ってローカルで実行できます。任意の第3章については、このノートブックを [qBraid](https://qbraid.com/) または [Google Colab](https://colab.research.google.com/) で開いてください。 |

<br>


## インストール

まずこのセルを実行して、Python のバージョンを確認してください。

In [ ]:
import sys

recommended = (3, 10)
current = sys.version_info[:2]

if current < recommended:
    print(f"Python {current[0]}.{current[1]} を使用しています。Python {recommended[0]}.{recommended[1]} 以降を推奨します。")
else:
    print(f"Python {current[0]}.{current[1]} — 問題なさそうです！")


次に、下のセルを実行して必要なパッケージをインストール・アップグレードします。アップグレードの過程で、ローカル環境にすでにインストールされているパッケージとの依存関係の競合が生じる場合は、__新しい Python 環境__（Python 3.12 推奨）を作成し、そこでチャレンジを続けてください。

クリーンな環境のセットアップと Qiskit のインストールについては、IBM Quantum のドキュメントを参照してください: https://quantum.cloud.ibm.com/docs/guides/install-qiskit

In [ ]:
# 必要なパッケージをインストールします。
# このノートブックを初めて実行するときは、下の行のコメントを外して依存関係をインストールし、
# 以降の実行では再びコメントアウトしてください。


%pip install --upgrade 'qiskit[visualization]>=2.5.0' qiskit-aer qiskit-ibm-runtime matplotlib
%pip install --upgrade qc-grader

ライブラリのインストールとアップグレードが完了したら、Python カーネルを再起動し、下の「ライブラリのインポート」セルからノートブックを再実行してください。

In [ ]:
import os
import platform
import shutil
import subprocess
import sys

import qc_grader
import qiskit

from qc_grader.challenges.qgss_2026 import check_progress
from qc_grader.challenges.qgss_2026.lab0 import (
    grade_lab0_ex1,
    grade_lab0_ex2,
    grade_lab0_ex3,
    grade_lab0_ex4,
)


from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import array_to_latex, plot_histogram, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Batch, QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

import matplotlib.pyplot as plt

print(f"Qiskit version: {qiskit.__version__}")
print(f"Grader version: {qc_grader.__version__}")



# 第1章 — Qiskit パターンによる初めての量子プログラム

## 1.1 IBM Quantum アカウント

まず最初に！本編に参加するには、有効な IBM Quantum アカウントが必要です。

すでに `QiskitRuntimeService` でアカウント認証情報を保存している場合は、下のセルを実行するだけでバックエンドの一覧が表示されます。新規ユーザーの場合や、前回セットアップしてからしばらく経っている場合は、次の手順でアカウントを保存してください。

1. [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com/) にアクセスしてサインインします（またはアカウントを作成します）。
2. ダッシュボードで `Create +` または `View all` をクリックして API キーをコピーします。
3. Instances タブからインスタンスの CRN をコピーします。
4. 下のセルのプレースホルダーを自分の API キーとインスタンス CRN に置き換えて、実行します。

完了したら、grader のセルを実行して、認証情報が正しく設定されていることを確認してください。

In [ ]:
# ここに IBM Cloud の認証情報を貼り付けます。環境ごとに一度だけ行えば十分です。
# 動作を確認したら、認証情報がファイルに残らないようこれらの行をコメントアウトできます。

#token = "<YOUR_API_KEY>"
#instance = "<YOUR_CRN>"  # "crn:v1:bluemix:public:quantum-computing:..." のような形式です

#QiskitRuntimeService.save_account(
#    token=token,
#    instance=instance,
#    overwrite=True,
#    set_as_default=True,
#)

# 確認
service = QiskitRuntimeService()
backends = service.backends()
print(f"アカウントは正常です。{len(backends)} 個のバックエンドが利用可能です:")
for b in backends[:5]:
    print(f"  {b.name} ({b.num_qubits} qubits)")


<div class="alert alert-block alert-success">
  <b>演習1: IBM Quantum アカウントの確認</b>

  API キーと CRN を設定し、利用可能なバックエンドを確認しましたか？
  下の演習1のセルを実行して、grader が正しく動作することを確認してください。
  何も入力する必要はありません！

  アカウントが正しく設定されていれば、成功メッセージが表示されます。
</div>


In [ ]:
grade_lab0_ex1()

## 1.2 ゲートによる量子計算

IBM の量子コンピューターはゲートベースです。古典コンピュータがビットを処理するのに AND、OR、NOT ゲートを使うのとよく似て、私たちは量子ゲートを組み合わせて量子アルゴリズムを構築します。大きな違いは、量子ゲートが量子ビット（古典ビットの量子版）に作用することです。古典ビットは厳密に 0 か 1 のどちらかですが、量子ビットは3つの量子力学的な現象を示せます。すなわち、重ね合わせ（測定されるまで量子ビットは 0 と 1 の線形結合として存在する）、量子もつれ（2つ以上の量子ビットが、個々の量子ビット単独では記述できない共同の量子状態を共有する）、干渉（量子振幅は強め合ったり打ち消し合ったりでき、これによりアルゴリズムは誤った答えを抑制し正しい答えを強化できる）です。これら3つの現象こそが、量子計算を古典計算と異なるものにしています。

### 重ね合わせとアダマールゲート

[アダマールゲート (H)](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.HGate) は、確定した状態（$|0\rangle$ または $|1\rangle$）にある量子ビットを、両者の等しい重ね合わせにします。

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \qquad H|0\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr)$$

この状態を測定すると、0 か 1 が等しい確率（それぞれ50%）で得られます。$\frac{1}{\sqrt{2}}$ の係数は確率振幅であり、各結果の測定確率はその振幅の2乗です。

下のセルを完成させて、アダマールゲートを備えた初めての1量子ビット回路を構築しましょう。

<br>

<div class="alert alert-block alert-info">

重ね合わせ、量子もつれ、干渉についてさらに学ぶには、[量子情報の基礎](https://quantum.cloud.ibm.com/learning/courses/basics-of-quantum-information) コースを参照してください。

</div>


<div class="alert alert-block alert-success">
<b>演習2: 初めての量子回路を構築する</b>

下は空の1量子ビット回路です。量子ビット0に <b>アダマールゲート</b> を追加して、重ね合わせにしましょう。これが、1.6節で Sampler を実行したときに特徴的な 50/50 の分布を生み出すものです。

ヒント: 公式の [クイックスタート](https://quantum.cloud.ibm.com/docs/guides/quick-start) を試してみてください。

</div>


In [ ]:

qc = QuantumCircuit(1)

# TODO: 量子ビット0にアダマールゲートを追加してください

qc.draw("mpl")


## 1.3 Qiskit パターンとは？

これで $H|0\rangle$ の重ね合わせ状態を準備する量子回路ができました。次の節では、この回路で2つの実験を行います。1つは測定カウントを収集する実験、もう1つは Z 観測量の期待値を計算する実験です。どちらの実験も、[Qiskit パターン](https://quantum.cloud.ibm.com/docs/guides/intro-to-patterns) と呼ばれる同じ4ステップのワークフローに従います。

| ステップ | 行うこと | このラボでは |
|------|------------|-------------|
| 1. マップ (Map) | 問題を量子回路と演算子に対応づける。 | H 回路を構築し、Z 観測量を定義します。 |
| 2. 最適化 (Optimize) | 対象ハードウェア向けに回路を最適化する。 | `generate_preset_pass_manager` を使って行います。 |
| 3. 実行 (Execute) | プリミティブを使って対象ハードウェア上で実行する。 | Sampler と Estimator の両方で回路を実行します。 |
| 4. 後処理 (Post-process) | 結果を後処理する。 | 測定ヒストグラムを描画し、⟨Z⟩ ≈ 0 を確認します。 |

実行ステップのために、Qiskit は2つのプリミティブを提供します。

- [SamplerV2](https://quantum.cloud.ibm.com/docs/guides/primitives) は、量子回路の実行による出力レジスタをサンプリングします。これを使って測定カウントを収集し、0 と 1 がほぼ等しい頻度で現れることを確認します。
- [EstimatorV2](https://quantum.cloud.ibm.com/docs/guides/primitives) は、量子回路が準備した状態に関する観測量の期待値を計算します。これを使って $\langle Z \rangle$ を評価します。ここで $|0\rangle$ は固有値 $+1$、$|1\rangle$ は固有値 $-1$ を持ち、等しい重ね合わせに対しては $\langle Z \rangle = 0$ となります。

次の節では各ステップを順に見ていきます。

## 1.4 ステップ1 — マップ (Map)

マップのステップでは、量子問題を回路と観測量として定義します。

回路はすでに構築済みです。`qc` は $H|0\rangle$ 状態を準備します。ここでは観測量を定義します。$|0\rangle$ に対して固有値 $+1$、$|1\rangle$ に対して $-1$ を持つ Z 演算子（パウリ Z）を使います。

$$Z = |0\rangle\langle 0| - |1\rangle\langle 1|$$

$H|0\rangle$ 状態では両方の結果が等しく起こりうるので、期待値は次のようになります。

$$\langle Z \rangle = \frac{1}{2}(+1) + \frac{1}{2}(-1) = 0$$

これを実行ステップで Estimator を使って確認します。

<div class="alert alert-block alert-success">
<b>演習3: 観測量を定義する</b>

マップのステップでは、何を測定するかを選びます。ここでは量子ビット0上のパウリ Z 演算子が必要です。その期待値は、量子ビットが |0⟩ と |1⟩ のどちらにどれだけ偏っているかを教えてくれます。

係数 $1.0$ を持つ単一量子ビットの Z __観測量__ を <code>SparsePauliOp</code> を使って作成してください。

ヒント: [SparsePauliOp のドキュメント](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp) に良い例があります。

</div>


In [ ]:

# TODO: 観測量を単一量子ビットの Z 演算子として定義してください
observable = None

print(f"Observable: {observable}")


## 1.5 ステップ2 — トランスパイル (Transpile)

量子ハードウェアには制約があります。実際の量子ハードウェアは、その基盤となる物理的実装によって決まるネイティブゲートセットと量子ビットの接続性を持ちます。`トランスパイル` のステップは、抽象的な回路を、対象バックエンドが実際に実行できる等価な回路へと、そのサポートするゲートセットと接続性を使って書き換えます。

これを処理するために `generate_preset_pass_manager` を使います。下のセルでは、ほぼどんなゲートも受け付ける理想的なシミュレーターを対象とするので、トランスパイル後の回路は元の回路に近い見た目になります。任意の 1.8 節では、トランスパイラーがより多くの作業を行う実機上で実行します。すなわち、ゲートをバックエンドのネイティブゲートセットに分解し、接続性の制約に従って回路レイアウトを最適化します。

トランスパイルは回路と観測量の両方に適用されます。トランスパイラーが仮想量子ビットを物理量子ビットに割り当てるとき、観測量も同じ対応づけを反映しなければなりません。これを `apply_layout` で行います。今回の回路は量子ビットが1つだけなのでレイアウトは自明ですが、実機上のマルチ量子ビット回路でも同じパターンが必要になります。

In [ ]:
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

# Sampler は回路内に測定を必要とします
qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm.run(qc_meas)

# Estimator は測定なしの回路を受け取り、観測量を別途評価します
isa_qc = pm.run(qc)

# トランスパイル後の回路と量子ビットのインデックスが一致するよう、観測量に同じレイアウトを適用します
isa_observable = observable.apply_layout(isa_qc.layout)

print("トランスパイル後の回路と観測量の準備ができました。")

## 1.6 ステップ3 — 実行 (Execute)

いよいよトランスパイル後の回路を両方のプリミティブを使って実行します。それぞれが同じ量子状態について異なる問いに答えます。

### Sampler: 「どんなカウントが得られる？」

Sampler は回路を何度も（ショット）実行し、各結果がどれくらいの頻度で起こるかを数えます。すなわち、0 は何回測定され、1 は何回測定されるか、です。

In [ ]:
sampler = Sampler(mode=backend)
sampler_result = sampler.run([isa_qc_meas], shots=1000).result()
counts = sampler_result[0].data.meas.get_counts()

print("Sampler counts:", counts)

<div class="alert alert-block alert-success">
<b>Sampler の結果を提出・確認して演習2を完了する</b>

たった今計算した <code>counts</code> は、1.2 節で構築した H 回路から得られたものです。下のセルを実行して提出してください。grader が 50/50 の分布を確認します。

</div>


In [ ]:
grade_lab0_ex2(counts)

### Estimator: 「量子回路を観測したときにどんな値が得られる？」

Estimator は回路と観測量を受け取り、カウントを手作業で収集・平均化することなく、期待値を直接返します。回路に測定ゲートは不要です。Estimator が内部で測定を処理します。

$H|0\rangle$ 状態については、$\langle Z \rangle = 0$ が期待されます。


In [ ]:
# Estimator を実行 — 測定なしの回路 + 観測量
estimator = Estimator(mode=backend)
estimator_result = estimator.run([(isa_qc, isa_observable)]).result()
exp_val = estimator_result[0].data.evs

print(f"⟨Z⟩ = {exp_val:.4f}")

<div class="alert alert-block alert-success">
<b>Estimator の結果を提出・確認して演習3を完了する</b>

下のセルを実行して <code>exp_val</code> を grader に提出してください。完璧な H|0⟩ の重ね合わせであれば、⟨Z⟩ は ≈ 0 になるはずです。

</div>


In [ ]:
grade_lab0_ex3(exp_val)

## 1.7 ステップ4 — 後処理 (Post-process)

後処理のステップは、生の結果から意味を引き出すところです。問題に応じて、測定分布の可視化、期待値から派生量を計算すること、結果を古典的な最適化ループに渡すこと、あるいは誤り抑制（エラー緩和）技術を適用することなどを意味します。

このラボでは単純に保ちます。Sampler のカウントをヒストグラムとして描画して 0 と 1 が何回測定されたかを見て、Estimator の結果を、$\langle Z \rangle$ の統計的不確かさを示す誤差棒付きの棒グラフとして描画します。シミュレーター上では等しいカウントと $\langle Z \rangle \approx 0$ が期待されます。実機上では、どちらのグラフもショットノイズとデバイスノイズの影響を反映します。

In [ ]:

# Sampler の結果
print(f"Sampler counts: {counts}")
display(plot_histogram(counts, title="H|0⟩ measurement distribution (1000 shots)"))

# Estimator の結果
ev = estimator_result[0].data.evs.item()
std = estimator_result[0].data.stds.item()
print(f"\nEstimator ⟨Z⟩ = {ev:.4f} ± {std:.4f}")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["⟨Z⟩"], [ev], yerr=[std], capsize=10, color="steelblue", width=0.4)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel("Expectation value")
ax.set_title("Z observable on H|0⟩ (Estimator)")
plt.tight_layout()
plt.show()


## 1.8 (任意) 実際の量子ハードウェアで実行する

<div class="alert alert-block alert-info">
<b>この節は任意であり、採点対象ではありません。</b> 実際の量子コンピューターにジョブを投入し、あなたの QPU 時間を使用します。スキップして第2章に進んでも構いません。
</div>

ここまではシミュレーターを使ってきました。今度は同じ回路を実際の IBM 量子プロセッサ上で実行します。コードは同じ Qiskit パターンのワークフローに従います。唯一の変更点はバックエンドです。

実際の量子ハードウェアは、シミュレーターにはない物理的制約の下で動作します。ゲートは物理量子ビット上のマイクロ波パルスとして実行され、各操作の忠実度は有限です。その結果、カウントは 50/50 に近くなりますが正確ではなく、$\langle Z \rangle$ は測定可能な不確かさを伴って 0 に近い値になります。これらの結果をシミュレーターと比較すると、量子誤り抑制がどこで役立つかを具体的に実感できます。

Sampler と Estimator のジョブを [Batch](https://quantum.cloud.ibm.com/docs/guides/run-jobs-batch) を使って一緒に投入します。Batch は複数のジョブを単一のバックエンド予約にまとめ、再キューイングなしで連続して実行させます。このジョブは `Heron` デバイスで約20秒の QPU 時間を使用します。キューの待ち時間はバックエンドの負荷によって変わります。ジョブ ID が表示される前にセルがタイムアウトした場合は、次のセルでジョブ ID から結果を取得する方法を示します。

In [ ]:
# デフォルトでは、この節は実際の量子ハードウェア上で実行されます。
# 本編のために QPU 時間を節約したい場合は、下のノイズ入りシミュレーターの行のコメントを外し、
# この節の残りで real_backend を real_backend_sim に置き換えてください。

real_backend = service.least_busy()
# real_backend_sim = AerSimulator.from_backend(real_backend)  # ノイズ入りシミュレーター

print(f"Running on: {real_backend.name} ({real_backend.num_qubits} qubits)")

# ステップ2: 対象バックエンド向けにトランスパイル
pm_real = generate_preset_pass_manager(optimization_level=1, backend=real_backend)

qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm_real.run(qc_meas)

isa_qc_real = pm_real.run(qc)
isa_obs = observable.apply_layout(isa_qc_real.layout)

# ステップ3: 実行 — 再キューイングを避けるため両方のプリミティブを単一の Batch で実行します
with Batch(backend=real_backend):
    sampler = Sampler()
    sampler.options.environment.job_tags = ["qgss26"]
    sampler_job = sampler.run([isa_qc_meas], shots=1000)

    estimator = Estimator()
    estimator.options.environment.job_tags = ["qgss26"]
    estimator_job = estimator.run([(isa_qc_real, isa_obs)])

sampler_job_id = sampler_job.job_id()
estimator_job_id = estimator_job.job_id()

print(f"Sampler job ID:   {sampler_job_id}")
print(f"Estimator job ID: {estimator_job_id}")


In [ ]:
# (任意) 上のセルが結果を表示する前にタイムアウトした場合は、その出力から
# Sampler/Estimator のジョブ ID を下の文字列にコピーして、このセルを実行してください。
# そうでない場合はコメントアウトしたままにしてください。

# sampler_job_id = "<paste sampler job id>"
# estimator_job_id = "<paste estimator job id>"
# sampler_job = service.job(sampler_job_id)
# estimator_job = service.job(estimator_job_id)


In [ ]:
# ステップ4: 結果
hw_counts = sampler_job.result()[0].data.meas.get_counts()
hw_estimator_result = estimator_job.result()
hw_exp_val = hw_estimator_result[0].data.evs

print(f"Hardware Sampler counts: {hw_counts}")
print(f"Hardware Estimator ⟨Z⟩:  {hw_exp_val:.4f}")

結果を並べて比較しましょう。シミュレーターは 50/50 の分布を与えます。ハードウェアの結果はノイズの影響、すなわちそこからの小さなずれを示します。

In [ ]:
display(plot_histogram(
    [counts, hw_counts],
    legend=["Simulator", f"Hardware ({real_backend.name})"],
    title="H|0⟩ — Simulator vs Real Hardware",
))


# Estimator の結果
ev = hw_estimator_result[0].data.evs.item()
std = hw_estimator_result[0].data.stds.item()
print(f"\nEstimator ⟨Z⟩ = {ev:.4f} ± {std:.4f}")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["⟨Z⟩"], [ev], yerr=[std], capsize=10, color="steelblue", width=0.4)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel(f"Expectation value from ({real_backend.name})")
ax.set_title("Z observable on H|0⟩ (Estimator)")
plt.tight_layout()
plt.show()


# 第2章 — マルチ量子ビット回路と可視化

第1章、お疲れさまでした！単一量子ビットの重ね合わせを構築し、それを Sampler と Estimator に通して、同じ量子状態が2つの異なる測定の視点からどう見えるかを、シミュレーター上で、そして試したなら実際の量子ハードウェア上でも見てきました。

さて、もっと面白いものを構築しましょう。この章では、6個の量子ビットを扱って、特別な量子回路を一段階ずつ組み立てます。最後に、それが生成する状態ベクトルを見て、[QSphere](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.visualization.plot_state_qsphere) を使って可視化します。QSphere は各基底状態の確率と位相を示すマルチ量子ビット可視化ツールです。最終的な状態が何に見えるか、あなたにはわかるかもしれません。

## 2.1 6量子ビットのもつれ状態を構築する

この演習では3つのゲートを使います。

- [H](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.HGate): 量子ビットを $|0\rangle$ と $|1\rangle$ の等しい重ね合わせにします（これはもうご存じですね）。
- [X](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.XGate): 量子ビットを $|0\rangle$ から $|1\rangle$ へ、またはその逆へ反転させます。古典的な NOT ゲートの量子版です。
- [CX](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.CXGate): 制御量子ビットが $|1\rangle$ のとき、かつそのときに限り標的量子ビットを反転させる2量子ビットゲートです。重ね合わせ状態に CX を適用すると、2つの量子ビットの間に量子もつれが生まれます。

<div class="alert alert-block alert-info">

これらのゲートをさらに探求し、実際のハードウェアで実行したい場合は、<a href="https://quantum.cloud.ibm.com/learning/courses/use-a-qc-today/build-and-run-your-first-quantum-program">Build and run your first quantum program</a> を参照してください。

</div>


すべての量子ビットは $|0\rangle$ で始まります。下のセルから始めましょう。


<div class="alert alert-block alert-success">
<b>演習4: 6量子ビットのもつれ状態を構築する</b>

6量子ビットの回路を構築します。後で QSphere 上で可視化すると、見覚えのあるものが見えるでしょう。

下のセルを使って3つのステップで構築します。

1. （すでに与えられています）6量子ビットの回路を作成します。
2. 量子ビット0に <b>H</b> を適用して重ね合わせにし、次に量子ビット1に <b>X</b> を適用します。
3. 量子ビット0を制御、量子ビット1から5を標的とする <b>CNOT チェーン</b> を適用して、すべてをもつれさせます。

ヒント: `for` ループを使って CNOT チェーンを構築できます。
</div>

In [ ]:
# ステップ1: 6量子ビットの回路を作成
special_qc = QuantumCircuit(6)
special_qc.draw("mpl")

次に、量子ビット0に H を適用して重ね合わせにします。続いて量子ビット1に X を適用し、$|0\rangle$ から $|1\rangle$ へ反転させます。これで回路には特定の状態に準備された2つの量子ビットができ、次のステップでの量子もつれの準備が整います。

In [ ]:
# ステップ2: 量子ビット0に H、量子ビット1に X
# TODO: 量子ビット0に H を適用してください

# TODO: 量子ビット1に X を適用してください


special_qc.draw('mpl')


次に、CNOT チェーンを使って量子もつれを作ります。各 CNOT ゲートは量子ビット0を制御として使い、量子ビット1から5を標的とします。これによりすべての量子ビットが連結されます。量子ビット0が重ね合わせにあるとき、他のすべての量子ビットがそれともつれます。回路図には、量子ビット0を他のすべての量子ビットに接続する5つの CNOT ゲートが表示されます。

In [ ]:
# ステップ3: CNOT チェーン — すべての量子ビットを量子ビット0ともつれさせる
# TODO: 量子ビット1から5にループし、量子ビット0を制御として CNOT を適用してください


special_qc.draw('mpl')


## 2.2 状態ベクトル

状態ベクトルは、量子状態の完全な数学的記述です。$n$ 量子ビットに対して、それは $2^n$ 個の振幅からなる複素ベクトルで、各振幅は可能な各基底状態に1つずつ対応します。特定の基底状態を測定する確率は、その振幅の2乗です。

私たちの6量子ビット回路では、状態ベクトルは $2^6 = 64$ 個の要素を持ち、$|000000\rangle$ から $|111111\rangle$ までのすべての基底状態を網羅します。`Statevector` クラスがこれを回路から計算し、`array_to_latex` がそれを数学的な形式で表示します。

ほとんどの要素はゼロになります。振幅がゼロでない基底状態だけが、あなたが構築した量子状態に寄与します。

In [ ]:
sv = Statevector(special_qc)
array_to_latex(sv, max_size=64)

## 2.3 量子状態を可視化する

64個の複素数を直接読むのは実用的ではありません。構造を一目で見る方法が必要です。

単一量子ビットについては、[ブロッホ球](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.visualization.plot_bloch_multivector) が任意の純粋状態を球面上の点に対応づけます。しかしこれは一度に1量子ビットしか扱えず、6つの別々のブロッホ球では量子ビットがどのようにもつれているかを示せません。

[QSphere](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.visualization.plot_state_qsphere) は、マルチ量子ビットの完全な状態を単一の球面上に表します。各基底状態は球面上の点で、3つの性質を持ちます。

- 緯度はハミング重み（ビット列に含まれる1の個数）で決まります。1が0個の状態は北極に、1がすべての状態は南極に位置します。
- 点の大きさは、その基底状態の測定確率を表します。振幅がゼロの状態は表示されません。
- 色は量子位相を表します。

あなたが構築した状態については、同じ大きさの2つの点が見えるはずです。それらがどの基底状態に対応するか見てみましょう。

In [ ]:
# QSphere
plot_state_qsphere(sv)

### QSphere が示すもの

QSphere には2つの点があり、1つは上部近く、もう1つは下部近くにあります。それらは、量子もつれで結ばれた、等しい重ね合わせにある2つの基底状態です。

<div class="alert alert-block alert-success">
<b>演習4 — 状態ベクトルを提出する</b>

<code>sv</code> を grader に提出して演習4を完了してください。grader はあなたの回路を確認し、なぜこの状態が特別なのかを明らかにします。また、本編に向けた準備に役立つコースへのリンクも受け取れます。下のセルを実行して Lab 0 を締めくくりましょう。

</div>


In [ ]:
grade_lab0_ex4(sv)


### 次に来るもの

IBM は2016年5月4日に最初の量子コンピューターをクラウドに公開しました。それ以来、Qiskit と量子計算コミュニティーは、ハードウェア、ソフトウェア、教育の各世代を通じて共に成長してきました。Qiskit はそのコミュニティーによって作られており、あなたの量子の旅の間ずっと寄り添い続けます。

このラボでは、シミュレーターと実機で量子回路を実行し、2つの異なるプリミティブで量子状態を測定し、6量子ビットのもつれ状態を可視化しました。これらは、QGSS 2026 を通して使う中核となる構成要素です。

進捗を確認して、主要な演習を締めくくりましょう。第3章には任意の演習があり、Qiskit を使うもう一つの方法——C で量子回路を構築すること——を試せます。準備ができたら立ち寄ってください。

In [ ]:
check_progress("lab0")

# 第3章 (任意、採点対象外) — C で量子回路を構築する

<div class="alert alert-block alert-info">
<b>この章は任意であり、採点対象ではありません。</b> C コンパイラ（GCC または Clang）が必要です。Windows ユーザーは、始める前に 3.1 節を参照してください。
</div>

前の章では Python で量子回路を構築しました。Qiskit は [C API](https://www.ibm.com/quantum/blog/c-api-enables-end-to-end-hpc-demo) も公開しており、C で回路を構築して Python に渡すことができます。これは、量子プログラムが、コンパイル型言語で書かれた古典的な高性能計算（HPC）ワークロードと並んで動作する場合に役立ちます。

この章は [Extend Qiskit in Python with C](https://quantum.cloud.ibm.com/docs/guides/c-extension-for-python) ガイドに従います。第2章の回路を C で再構築し、次に Python を使って状態ベクトルを計算します。

<div class="alert alert-block alert-warning">
Qiskit C API は実験的であり、ビルドに使うバージョンがインストール済みランタイムのマイナーバージョンと一致している必要があります。このノートブックは最新の Qiskit バージョンをインストールするので、ビルドとランタイムは常に同期しています。詳細は IBM の <a href="https://quantum.cloud.ibm.com/docs/guides/c-extension-for-python">C 拡張ガイド</a> を参照してください。
</div>


## 3.1 セットアップ

Qiskit C API のヘッダーとライブラリは、pip で Qiskit をインストールすると含まれるので、別途ダウンロードする必要はありません。C 拡張をビルドするには、C コンパイラと `setuptools` が必要です。

macOS、Linux、Google Colab、qBraid では、すべてそのまま動作します。下のセットアップセルを実行して始めましょう。

<details>
<summary><b>Windows ユーザーの方へ</b></summary>

<br>

第3章は Windows では公式にはサポートされていません。Windows 環境は、コンパイラのツールチェーン、MSVC のバージョン、BLAS の有無、パスの扱いが大きく異なるため、単一のノートブックですべての構成にわたって円滑なビルドを保証することはできません。

摩擦のない体験を得るには、C コンパイラがすでに利用可能な <a href="https://qbraid.com/">qBraid</a> または <a href="https://colab.research.google.com/">Google Colab</a> でこのノートブックを開いてください。

</details>

<div class="alert alert-block alert-info">
すでに GCC または Clang をインストールしていて、手動でセットアップしたい場合は、セットアップセルをスキップして IBM のドキュメントに直接従うことができます。

<ul>
<li><a href="https://quantum.cloud.ibm.com/docs/guides/install-c-api">Install the Qiskit C API</a></li>
<li><a href="https://quantum.cloud.ibm.com/docs/guides/c-extension-for-python">Extend Qiskit in Python with C</a></li>
<li><a href="https://quantum.cloud.ibm.com/docs/api/qiskit-c">Qiskit C API reference</a></li>
</ul>
</div>

In [ ]:
import sys
import os
import platform
import shutil
import subprocess

from qiskit.quantum_info import Statevector
from qiskit.visualization import array_to_latex, plot_state_qsphere

# macOS / Linux / Colab / qBraid 向けのセットアップ。
# Windows では、3.1 節の注記を参照し、qBraid または Google Colab を使用してください。

if platform.system() == "Windows":
    print("Windows: この章は qBraid または Google Colab で実行してください。")
else:
    IS_COLAB = "google.colab" in sys.modules

    # Colab にはデフォルトでコンパイラがありません。
    if IS_COLAB and shutil.which("gcc") is None:
        subprocess.run(["apt-get", "-qq", "install", "-y", "build-essential"], check=True)

    # setuptools は C ファイルを Python モジュールにビルドします。
    try:
        import setuptools  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools"], check=True)

    cc = shutil.which("gcc") or shutil.which("clang")
    if cc is None:
        print("C コンパイラが見つかりません。macOS では次を実行してください: xcode-select --install")
    else:
        print("C コンパイラ:", cc)
        print("セットアップ完了。3.2 に進んでください。")

## 3.2 ロゴ回路を C で構築する

この演習で使う C API 関数:

| 関数 | 何をするか |
|---|---|
| `qk_circuit_new(num_qubits, num_clbits)` | 新しい回路を作成します |
| `qk_circuit_gate(qc, QkGate_H, qubits, NULL)` | ゲートを追加します。`qubits` は量子ビットインデックスの `uint32_t` 配列です。`QkGate_H`、`QkGate_X`、`QkGate_CX` を使います。 |
| `qk_circuit_measure(qc, qubit, clbit)` | 測定を追加します |
| `qk_circuit_to_python_full(qc)` | C の回路を Python の `QuantumCircuit` に変換し、所有権を移します。この後に `qk_circuit_free` を呼ばないでください。 |

Qiskit C API を使うすべての C 拡張に適用される2つのルール:

- `QISKIT_PYTHON_EXTENSION` を定義し、`qiskit.h` の前に `Python.h` をインクルードします。
- `PyInit_<name>` の中で `qk_import()` を一度呼び出します。これがないと、API 呼び出しは実行時に失敗します。

完全なボイラープレートを含む一通りの実例については、[C 拡張ガイド](https://quantum.cloud.ibm.com/docs/guides/c-extension-for-python) を参照してください。

<div class="alert alert-block alert-info">
Python はコンパイル済みモジュールをセッションごとに一度だけ読み込みます。C ファイルを編集して再ビルドした場合は、再度インポートする前にカーネルを再起動してください。
</div>


<div class="alert alert-block alert-success">
<b>任意の演習: ロゴ回路を C で構築する</b>

演習4と同じ6量子ビット回路を、今度は C で構築します。下のファイルの3つのゲートステップを埋めてください。

1. 量子ビット0に <b>H</b> を適用します。
2. 量子ビット1に <b>X</b> を適用します。
3. <b>CNOT チェーン</b> を追加します。制御は量子ビット0、標的は量子ビット1から5です。ここでは <code>for</code> ループが使えます。

この回路には測定がないので、次のステップでその状態ベクトルを計算できます。
</div>

In [ ]:
%%writefile qgss_circuit.c

#include <Python.h>
#include <qiskit.h>
#include <stdint.h>

static PyObject *build_special_circuit(PyObject *self, PyObject *args) {
    QkCircuit *qc = qk_circuit_new(6, 0);

    // TODO 1: 量子ビット0に H を適用してください。
    //   QkGate_H を指定して qk_circuit_gate を使います。C API リファレンスを参照してください。

    // TODO 2: 量子ビット1に X を適用してください。
    //   QkGate_X を指定して qk_circuit_gate を使います。

    // TODO 3: CNOT チェーンを追加してください。制御は量子ビット0、標的は1から5です。
    //   各標的について、QkGate_CX と {制御, 標的} の配列を指定して qk_circuit_gate を呼び出します。
    //   ここでは for ループが使えます。

    return qk_circuit_to_python_full(qc);
}

static PyMethodDef methods[] = {
    {"build_special_circuit", build_special_circuit, METH_NOARGS, "Build the special circuit."},
    {NULL, NULL, 0, NULL},
};
static struct PyModuleDef moduledef = {
    .m_base = PyModuleDef_HEAD_INIT,
    .m_name = "qgss_circuit",
    .m_methods = methods,
};
PyMODINIT_FUNC PyInit_qgss_circuit(void) {
    if (qk_import() < 0) {
        return NULL;
    }
    return PyModuleDef_Init(&moduledef);
}


TODO を埋めたら、下のセルを実行して拡張をビルドしてください。

In [ ]:
%%writefile setup.py
from setuptools import setup, Extension
import qiskit.capi
import sys
import subprocess

def supports_no_warn_duplicate():
    try:
        result = subprocess.run(
            ["ld", "-no_warn_duplicate_libraries"],
            capture_output=True
        )
        return b"unknown option" not in result.stderr
    except Exception:
        return False

extra_link_args = []
if sys.platform == "darwin" and supports_no_warn_duplicate():
    extra_link_args = ["-Wl,-no_warn_duplicate_libraries"]

setup(
    name="qgss_circuit",
    ext_modules=[
        Extension(
            name="qgss_circuit",
            sources=["qgss_circuit.c"],
            include_dirs=[qiskit.capi.get_include()],
            define_macros=[("QISKIT_PYTHON_EXTENSION", "1")],
            extra_link_args=extra_link_args,
        )
    ],
)


`setup.py build_ext --inplace` は C ファイルをコンパイルし、結果のモジュールを現在のディレクトリに配置して、インポートできる状態にします。

In [ ]:
# C ファイルを Python モジュールにビルドします。
subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], check=True)
print("ビルド完了。")
print("このセルを以前に実行したことがある場合は、インポートする前にカーネルを再起動してください (Kernel > Restart)。")

## 3.3 Python で状態を読む

Qiskit C API は回路を構築しますが、シミュレーションは行いません。状態ベクトルを得るには、回路を Python に戻し、第2章と同じツールを使います。

In [ ]:
import qgss_circuit

special_qc_c = qgss_circuit.build_special_circuit()
sv = Statevector(special_qc_c)
array_to_latex(sv, max_size=64)

In [ ]:
plot_state_qsphere(sv, show_state_labels=True)

第2章で示したのと同じ回路を構築したので、回路が正しく組み立てられていれば、最終的な状態ベクトルは grader の演習4に合格するはずです。
合格しなくても驚かないでください！多くの方はすでに先ほど grader に合格しているので、今回間違えても、最終的な進捗チェックの結果は変わりません。これは __任意__ の演習であることを忘れないでください。

In [ ]:
grade_lab0_ex4(sv)

In [ ]:
check_progress("lab0")

# 追加情報

**作成者:** Sophy Shin

**バージョン:** 1.0.0